# Q3. ER Stress and Reactive Glia

**Question**: How do ER stress and reactive glial states influence panglial network remodeling?

**Why this notebook exists**: Test whether ER stress, inflammatory cytokines, reactive astrocytes, and activated microglia are enriched early after demyelination and decline during Cx47 recovery.

**Relevant panels**: `panglial_connexins, er_stress_upr, inflammatory_cytokines, microglial_activation, astrocyte_identity`

**Recommended `.obs` keywords to look for**: `time, condition, cell, annotation, microglia, astro, cluster`

**Suggested datasets from the brief**: `lpc_remyelination, eae_ms_adaptive_oligo, cx47_cko_eae`

**Scope**: Core question.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display

CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next((path for path in CANDIDATES if (path / "src").exists() and (path / "config").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root. Start Jupyter from the repo root or notebooks directory.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for module_name in [name for name in list(sys.modules) if name == "cx47_oligo" or name.startswith("cx47_oligo.")]:
    del sys.modules[module_name]

from cx47_oligo.gene_panels import GENE_PANELS
from cx47_oligo.exports import ensure_results_dir, save_current_figure, save_json, save_table, save_text, slugify
from cx47_oligo.h5ad_tools import (
    adata_overview,
    dataset_catalog,
    discover_h5ad_files,
    expression_frame,
    load_h5ad,
    mean_scores_by_group,
    obs_column_summary,
    question_panel_table,
    suggest_obs_columns,
)
from cx47_oligo.notebook_viz import (
    composition_table,
    configure_notebook_style,
    plot_composition,
    plot_feature_heatmap,
    plot_gene_contribution,
    plot_group_counts,
    plot_scatter_relationship,
    plot_score_distributions,
    plot_score_heatmap,
    plot_top_ranked_genes,
    title_card_html,
)
from cx47_oligo.questions import QUESTION_BANK
from cx47_oligo.scanpy_tools import (
    available_question_marker_genes,
    assign_keyword_families,
    ensure_scanpy_adata,
    grouped_gene_expression,
    grouped_gene_expression_stats,
    obs_projection_frame,
    obs_keyword_mask,
    rank_genes_groups_df,
    score_question_panels_scanpy,
)

sc = configure_notebook_style(REPO_ROOT)


In [ ]:
display(
    title_card_html(
        "Q3. ER Stress and Reactive Glia",
        "How do ER stress and reactive glial states influence panglial network remodeling?",
        dataset_hint="Suggested default dataset: falcao_et_al",
    )
)


## Dataset And Metadata


In [ ]:
DATA_ROOT = REPO_ROOT / "data" / "raw"
LOCAL_H5AD_FILES = discover_h5ad_files(DATA_ROOT)
LOCAL_H5AD_FILES


In [ ]:
DATASET_PATH = REPO_ROOT / "data" / "raw" / "falcao_et_al.h5ad"
if not DATASET_PATH.exists():
    DATASET_PATH = LOCAL_H5AD_FILES[0] if LOCAL_H5AD_FILES else None
DATASET_PATH


In [ ]:
RESULTS_DIR = ensure_results_dir(REPO_ROOT, "03_er_stress_and_reactive_glia", DATASET_PATH)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR
print("Results will be written to:", RESULTS_DIR)
save_json(
    {
        "question_id": "q3",
        "question_title": "ER Stress and Reactive Glia",
        "dataset_path": str(DATASET_PATH) if DATASET_PATH else "",
        "group_columns": ["Group", "Renamed_clusternames"],
        "results_dir": str(RESULTS_DIR),
    },
    RESULTS_DIR / "run_metadata.json",
)


In [ ]:
adata = load_h5ad(DATASET_PATH) if DATASET_PATH else None
if adata is None:
    print("No .h5ad file found under data/raw yet. Add one or point DATASET_PATH to a local file.")
else:
    overview = adata_overview(adata)
    display(overview)
    save_table(overview, RESULTS_DIR / "adata_overview.csv", index=False)
    print("scanpy version:", sc.__version__)


In [ ]:
if adata is not None:
    suggested = suggest_obs_columns(adata)
    obs_summary = obs_column_summary(adata)
    panel_table = question_panel_table(adata, "q3")
    display(suggested)
    display(obs_summary.head(30))
    display(panel_table)
    save_table(suggested, RESULTS_DIR / "suggested_obs_columns.csv", index=False)
    save_table(obs_summary, RESULTS_DIR / "obs_column_summary.csv", index=False)
    save_table(panel_table, RESULTS_DIR / "question_panel_availability.csv", index=False)


## Scanpy Preparation


In [ ]:
GROUP_COLUMNS = ["Group", "Renamed_clusternames"]
adata_sc = None
results = None

if adata is not None:
    adata_sc = ensure_scanpy_adata(adata, repo_root=REPO_ROOT, copy=True)
    results = score_question_panels_scanpy(adata_sc, "q3", repo_root=REPO_ROOT)
    display(results["panel_summary"])
    if results["score_columns"]:
        display(adata_sc.obs[results["score_columns"]].head())
    save_table(results["panel_summary"], RESULTS_DIR / "panel_score_summary.csv", index=False)
    projection = obs_projection_frame(adata_sc, obs_columns=GROUP_COLUMNS, score_columns=results["score_columns"])
    save_table(projection, RESULTS_DIR / "cell_level_scores_umap.csv")
    save_text(
        "\n".join(
            [
                f"Question: How do ER stress and reactive glial states influence panglial network remodeling?",
                f"Dataset: {DATASET_PATH}",
                f"Results directory: {RESULTS_DIR}",
                f"Group columns: {', '.join(GROUP_COLUMNS)}",
                f"Score columns: {', '.join(results['score_columns'])}",
            ]
        ),
        RESULTS_DIR / "analysis_summary.txt",
    )


## Score Structure


In [ ]:
if adata_sc is not None and results is not None:
    for groupby in GROUP_COLUMNS:
        if groupby in adata_sc.obs.columns:
            print(f"Score means by {groupby}")
            grouped_scores = mean_scores_by_group(adata_sc.obs, results["score_columns"], groupby)
            display(grouped_scores)
            save_table(grouped_scores, RESULTS_DIR / f"score_means_by_{slugify(groupby)}.csv")


In [ ]:
PRIMARY_GROUP = next((column for column in GROUP_COLUMNS if adata_sc is not None and column in adata_sc.obs.columns), None)
SECONDARY_GROUP = next((column for column in GROUP_COLUMNS[1:] if adata_sc is not None and column in adata_sc.obs.columns), None)

if adata_sc is not None and results is not None and PRIMARY_GROUP is not None:
    group_counts = plot_group_counts(adata_sc.obs, PRIMARY_GROUP)
    display(group_counts)
    save_table(group_counts, RESULTS_DIR / f"cell_counts_by_{slugify(PRIMARY_GROUP)}.csv")
    save_current_figure(FIGURES_DIR / f"cell_counts_by_{slugify(PRIMARY_GROUP)}.png")

    score_heatmap = plot_score_heatmap(adata_sc.obs, PRIMARY_GROUP, results["score_columns"])
    display(score_heatmap)
    save_table(score_heatmap, RESULTS_DIR / f"score_heatmap_values_by_{slugify(PRIMARY_GROUP)}.csv")
    save_current_figure(FIGURES_DIR / f"score_heatmap_by_{slugify(PRIMARY_GROUP)}.png")

    score_dist = plot_score_distributions(adata_sc.obs, PRIMARY_GROUP, results["score_columns"])
    display(score_dist.head())
    save_table(score_dist, RESULTS_DIR / f"score_distributions_by_{slugify(PRIMARY_GROUP)}.csv", index=False)
    save_current_figure(FIGURES_DIR / f"score_distributions_by_{slugify(PRIMARY_GROUP)}.png")

    if SECONDARY_GROUP is not None:
        composition = plot_composition(adata_sc.obs, PRIMARY_GROUP, SECONDARY_GROUP)
        display(composition)
        save_table(composition, RESULTS_DIR / f"composition_{slugify(SECONDARY_GROUP)}_within_{slugify(PRIMARY_GROUP)}.csv")
        save_current_figure(FIGURES_DIR / f"composition_{slugify(SECONDARY_GROUP)}_within_{slugify(PRIMARY_GROUP)}.png")
        composition_counts = composition_table(adata_sc.obs, PRIMARY_GROUP, SECONDARY_GROUP, normalize="none")
        save_table(composition_counts, RESULTS_DIR / f"composition_counts_{slugify(SECONDARY_GROUP)}_within_{slugify(PRIMARY_GROUP)}.csv")
else:
    print("Set GROUP_COLUMNS so at least one valid grouping column is available.")


## Marker-Level Views


In [ ]:
PANEL_TO_INSPECT = QUESTION_BANK["q3"].panels[0]
MARKER_GENES = available_question_marker_genes(adata_sc, "q3", top_per_panel=3) if adata_sc is not None else []

if adata is not None:
    panel_expr = expression_frame(adata, GENE_PANELS[PANEL_TO_INSPECT])
    print(f"Matched {panel_expr.shape[1]} genes in {PANEL_TO_INSPECT}")
    display(panel_expr.head())
    save_table(panel_expr.head(100), RESULTS_DIR / f"panel_expression_preview_{PANEL_TO_INSPECT}.csv")
    save_text("\n".join(MARKER_GENES), RESULTS_DIR / "marker_genes_used.txt")


## Cell-Type-Specific Gene Expression


In [ ]:
GENE_BREAKDOWN_GROUP = next(
    (
        column
        for column in ["Renamed_clusternames", PRIMARY_GROUP, SECONDARY_GROUP]
        if column is not None and adata_sc is not None and column in adata_sc.obs.columns
    ),
    None,
)
GENE_BREAKDOWN_GENES = MARKER_GENES[: min(8, len(MARKER_GENES))]

if adata_sc is not None and GENE_BREAKDOWN_GROUP is not None and GENE_BREAKDOWN_GENES:
    gene_breakdown = grouped_gene_expression_stats(
        adata_sc,
        groupby=GENE_BREAKDOWN_GROUP,
        genes=GENE_BREAKDOWN_GENES,
    )

    display(gene_breakdown["n_cells"])
    save_table(gene_breakdown["n_cells"], RESULTS_DIR / f"gene_breakdown_cell_counts_by_{slugify(GENE_BREAKDOWN_GROUP)}.csv")
    save_text("\n".join(GENE_BREAKDOWN_GENES), RESULTS_DIR / f"gene_breakdown_genes_by_{slugify(GENE_BREAKDOWN_GROUP)}.txt")

    mean_expression = gene_breakdown["mean_expression"]
    pct_expressing = gene_breakdown["pct_expressing"]
    gene_contribution = gene_breakdown["gene_contribution"]

    display(mean_expression)
    save_table(mean_expression, RESULTS_DIR / f"marker_mean_expression_by_{slugify(GENE_BREAKDOWN_GROUP)}.csv")
    display(plot_feature_heatmap(mean_expression, title=f"Mean marker expression by {GENE_BREAKDOWN_GROUP}", cmap="crest"))
    save_current_figure(FIGURES_DIR / f"marker_mean_expression_by_{slugify(GENE_BREAKDOWN_GROUP)}.png")

    display(pct_expressing)
    save_table(pct_expressing, RESULTS_DIR / f"marker_pct_expressing_by_{slugify(GENE_BREAKDOWN_GROUP)}.csv")
    display(plot_feature_heatmap(pct_expressing, title=f"Percent of cells expressing markers by {GENE_BREAKDOWN_GROUP}", cmap="flare"))
    save_current_figure(FIGURES_DIR / f"marker_pct_expressing_by_{slugify(GENE_BREAKDOWN_GROUP)}.png")

    display(gene_contribution)
    save_table(gene_contribution, RESULTS_DIR / f"marker_gene_contribution_by_{slugify(GENE_BREAKDOWN_GROUP)}.csv")
    display(plot_gene_contribution(gene_contribution, title=f"Relative marker contribution within {GENE_BREAKDOWN_GROUP}"))
    save_current_figure(FIGURES_DIR / f"marker_gene_contribution_by_{slugify(GENE_BREAKDOWN_GROUP)}.png")

    sc.pl.dotplot(
        adata_sc,
        var_names=GENE_BREAKDOWN_GENES,
        groupby=GENE_BREAKDOWN_GROUP,
        standard_scale="var",
        dendrogram=False,
        save=f"_gene_breakdown_{slugify(GENE_BREAKDOWN_GROUP)}.png",
    )
    sc.pl.matrixplot(
        adata_sc,
        var_names=GENE_BREAKDOWN_GENES,
        groupby=GENE_BREAKDOWN_GROUP,
        standard_scale="var",
        cmap="mako",
        save=f"_gene_breakdown_{slugify(GENE_BREAKDOWN_GROUP)}.png",
    )
else:
    print("Need marker genes plus a valid grouping column such as Renamed_clusternames for cell-type-specific expression views.")


In [ ]:
if adata_sc is not None:
    color_columns = [column for column in GROUP_COLUMNS if column in adata_sc.obs.columns]
    color_columns += [column for column in results["score_columns"][:2] if column in adata_sc.obs.columns]
    if color_columns:
        sc.pl.umap(adata_sc, color=color_columns, ncols=2, save=f"_{slugify(PRIMARY_GROUP or 'group')}_overview.png")
    else:
        print("Set GROUP_COLUMNS to existing metadata columns before plotting UMAP.")


In [ ]:
FEATURE_UMAP_GENES = MARKER_GENES[: min(6, len(MARKER_GENES))]

if adata_sc is not None and FEATURE_UMAP_GENES:
    sc.pl.umap(adata_sc, color=FEATURE_UMAP_GENES, ncols=3, save=f"_{slugify(PRIMARY_GROUP or 'group')}_marker_genes.png")
else:
    print("Need marker genes present in the dataset for gene-level UMAP overlays.")


In [ ]:
if adata_sc is not None and PRIMARY_GROUP is not None and MARKER_GENES:
    sc.pl.dotplot(adata_sc, var_names=MARKER_GENES, groupby=PRIMARY_GROUP, standard_scale="var", dendrogram=False, save=f"_{slugify(PRIMARY_GROUP)}_markers.png")
    sc.pl.matrixplot(adata_sc, var_names=MARKER_GENES, groupby=PRIMARY_GROUP, standard_scale="var", cmap="mako", save=f"_{slugify(PRIMARY_GROUP)}_markers.png")
else:
    print("Need valid marker genes and a valid PRIMARY_GROUP for dotplot and matrixplot.")


In [ ]:
if adata_sc is not None and PRIMARY_GROUP is not None and results is not None and results["score_columns"]:
    sc.pl.violin(
        adata_sc,
        keys=results["score_columns"][: min(3, len(results["score_columns"]))],
        groupby=PRIMARY_GROUP,
        stripplot=False,
        rotation=90,
        save=f"_{slugify(PRIMARY_GROUP)}_panel_scores.png",
    )
else:
    print("Need valid score columns and a valid PRIMARY_GROUP for violin plots.")


## Differential Signals


In [ ]:
RANK_BY = next((column for column in GROUP_COLUMNS if adata_sc is not None and column in adata_sc.obs.columns), None)

if adata_sc is not None and RANK_BY is not None:
    rank_df = rank_genes_groups_df(adata_sc, groupby=RANK_BY, repo_root=REPO_ROOT, n_genes=10)
    display(rank_df.head(40))
    save_table(rank_df, RESULTS_DIR / f"rank_genes_by_{slugify(RANK_BY)}.csv", index=False)
    ranked_plot_df = plot_top_ranked_genes(rank_df, top_n=5)
    display(ranked_plot_df)
    save_table(ranked_plot_df, RESULTS_DIR / f"top_ranked_genes_by_{slugify(RANK_BY)}.csv", index=False)
    save_current_figure(FIGURES_DIR / f"top_ranked_genes_by_{slugify(RANK_BY)}.png")
else:
    print("Set GROUP_COLUMNS so at least one existing obs column is available for marker ranking.")


## Q3 Targeted Views

These cells focus on ER stress, inflammatory programs, and reactive glial-family structure in the Falcão-style EAE setting.


In [ ]:
Q3_CELL_COLUMN = next((column for column in ["Renamed_clusternames", "Celltypes", "leiden"] if adata_sc is not None and column in adata_sc.obs.columns), None)

if adata_sc is not None and Q3_CELL_COLUMN is not None and "Group" in adata_sc.obs.columns:
    q3_frame = adata_sc.obs[[Q3_CELL_COLUMN, "Group", *[column for column in ["er_stress_upr_score", "inflammatory_cytokines_score", "microglial_activation_score", "panglial_connexins_score"] if column in adata_sc.obs.columns]]].copy()
    q3_frame["cell_family"] = assign_keyword_families(
        q3_frame[Q3_CELL_COLUMN],
        {
            "Microglia": ["migl", "micro"],
            "Oligodendroglia": ["oligo", "opc", "cop", "imolg", "nfol", "mol"],
            "Astrocytes": ["astro"],
            "Vascular/VLMC": ["vlmc", "vascular", "endo", "pericyte"],
        },
    )

    relevant = [column for column in ["er_stress_upr_score", "inflammatory_cytokines_score", "microglial_activation_score", "panglial_connexins_score"] if column in q3_frame.columns]
    q3_summary = q3_frame.groupby(["Group", "cell_family"], observed=False)[relevant].mean()
    display(q3_summary)
    save_table(q3_summary, RESULTS_DIR / "q3_group_family_score_summary.csv")
    display(plot_feature_heatmap(q3_summary, title="ER stress and reactive glia by Group and family", cmap="mako"))
    save_current_figure(FIGURES_DIR / "q3_group_family_score_heatmap.png")

    family_comp = plot_composition(q3_frame, "Group", "cell_family")
    display(family_comp)
    save_table(family_comp, RESULTS_DIR / "q3_cell_family_composition_by_group.csv")
    save_current_figure(FIGURES_DIR / "q3_cell_family_composition_by_group.png")

    if "er_stress_upr_score" in q3_frame.columns and "microglial_activation_score" in q3_frame.columns:
        scatter_df = plot_scatter_relationship(
            q3_frame,
            x="er_stress_upr_score",
            y="microglial_activation_score",
            hue="Group",
            style="cell_family",
            title="ER stress vs microglial activation",
        )
        display(scatter_df.head())
        save_table(scatter_df, RESULTS_DIR / "q3_er_stress_vs_microglial_activation.csv", index=False)
        save_current_figure(FIGURES_DIR / "q3_er_stress_vs_microglial_activation.png")
else:
    print("Need a cluster-like annotation column and Group metadata for Q3-specific plots.")


## Interpretation Notes

This notebook is now scanpy-first. Once you identify useful metadata columns and candidate subpopulations, extend it with:

- `sc.pl.umap`, `sc.pl.dotplot`, `sc.pl.matrixplot`, or `sc.pl.violin`
- `sc.tl.rank_genes_groups` on lesion, condition, or cell-state groupings
- focused filtering to oligodendrocyte-lineage or glial subsets
- cross-dataset validation using the same panel definitions
